In [2]:
# =====================================================================
# الخطوة 1 & 2: تثبيت المكتبات القياسية المطلوبة لمعالجة الإشارات والبيانات
# =====================================================================
print("جاري تثبيت المكتبات الطبية والقياسية...")
!pip install -q wfdb numpy pandas scipy matplotlib

import os
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# تأكيد المجلدات المخصصة للبيانات الخام
raw_data_dir = '/content/mitdb_raw'
os.makedirs(raw_data_dir, exist_ok=True)
print("✓ تم تثبيت المكتبات وإعداد المجلدات.")

# =====================================================================
# الخطوة 3: تحميل قاعدة بيانات MIT-BIH بالكامل (48 سجلاً) من PhysioNet
# =====================================================================
print("\nجاري تحميل قاعدة البيانات الرسمية كاملة (48 سجلاً)...")

# القائمة الكاملة والمعتمدة لجميع المرضى في قاعدة البيانات
all_records = [
    100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
    111, 112, 113, 114, 115, 116, 117, 118, 119, 121,
    122, 123, 124, 200, 201, 202, 203, 205, 207, 208,
    209, 210, 212, 213, 214, 215, 217, 219, 220, 221,
    222, 223, 228, 230, 231, 232, 233, 234
]

# تحميل السجلات واحداً تلو الآخر لضمان الاستقرار التام وتفادي انقطاع السيرفر
for rec in all_records:
    rec_str = str(rec)
    # فحص ما إذا كان السجل محملاً مسبقاً لتوفير الوقت
    if os.path.exists(os.path.join(raw_data_dir, f"{rec_str}.hea")):
        continue

    try:
        print(f"-> جاري جلب السجل {rec_str}...")
        # تحميل الملفات الثلاثة الأساسية لكل سجل (.hea, .dat, .atr)
        wfdb.dl_database('mitdb', dl_dir=raw_data_dir, records=[rec_str], overwrite=True)
    except Exception as e:
        print(f"⚠️ خطأ أثناء تحميل السجل {rec_str}: {e}")

print("\n✓ اكتمل تحميل كافة السجلات الـ 48 الخام بنجاح في المجلد /content/mitdb_raw")

جاري تثبيت المكتبات الطبية والقياسية...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
✓ تم تثبيت المكتبات وإعداد المجلدات.

جاري تحميل قاعدة البيانات الرسمية كاملة (48 سجلاً)...
-> جاري جلب السجل 100...
Generating record list for: 100
Generating list of all files for: 100
Finished downloading files
-> جاري جلب السجل 101...
Generating record list for: 101
Generating list of all files for: 101
Finished downloading files
-> جاري جلب السجل 102...
Generating record list for: 102
Generating list of all files for: 102
Finished downloading fi

In [3]:
import os
import wfdb
import numpy as np
import pandas as pd

raw_data_dir = '/content/mitdb_raw'
all_records = [
    100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
    111, 112, 113, 114, 115, 116, 117, 118, 119, 121,
    122, 123, 124, 200, 201, 202, 203, 205, 207, 208,
    209, 210, 212, 213, 214, 215, 217, 219, 220, 221,
    222, 223, 228, 230, 231, 232, 233, 234
]

# 1. تعريف خريطة معايير AAMI EC57 الرسمية لرموز MIT-BIH
aami_mapping = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',  # مجموعة الطبيعي
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',            # فوق البطينية
    'V': 'V', 'E': 'V',                                # البطينية
    'F': 'F',                                          # الاندماج
    '/': 'Q', 'f': 'Q', 'Q': 'Q'                       # غير المعروف
}

beats_list = []
labels_list = []

print("جاري بدء معالجة السجلات وفقاً للمنهجية القياسية الموثقة...")

for rec in all_records:
    rec_str = str(rec)
    record_path = os.path.join(raw_data_dir, rec_str)

    try:
        # قراءة الإشارة الخام (القناة الأولى Lead II) والتوصيفات
        record = wfdb.rdrecord(record_path)
        signal = record.p_signal[:, 0]
        annotation = wfdb.rdann(record_path, 'atr')
    except FileNotFoundError:
        print(f"⚠️ تحذير: ملفات السجل {rec_str} غير موجودة. جاري تخطي هذا السجل.")
        continue
    except Exception as e:
        print(f"⚠️ تحذير: حدث خطأ غير متوقع عند معالجة السجل {rec_str}: {e}. جاري تخطي هذا السجل.")
        continue

    sample_locations = annotation.sample
    symbols = annotation.symbol

    # معالجة كل نبضة بشكل مستقل وعمياء
    for i in range(len(sample_locations)):
        sym = symbols[i]
        pos = sample_locations[i]

        if sym in aami_mapping:
            aami_class = aami_mapping[sym]

            # اقتطاع 150 عينة قبل القمة و 150 عينة بعدها (المجموع 300 عينة)
            if pos - 150 >= 0 and pos + 150 < len(signal):
                beat = signal[pos - 150 : pos + 150]

                if np.isnan(beat).any():
                    continue

                # التصنيف الثنائي
                if aami_class == 'N':
                    binary_label = 0
                elif aami_class in ['S', 'V', 'F']:
                    binary_label = 1
                else:
                    continue  # استبعاد فئة Q

                beats_list.append(beat)
                labels_list.append(binary_label)

# 2. تحويل البيانات إلى DataFrame
beats_array = np.array(beats_list)
labels_array = np.array(labels_list)

df_beats = pd.DataFrame(beats_array)
df_beats['label'] = labels_array

# 3. حفظ قاعدة البيانات الجديدة
output_dir = '/content/data'
os.makedirs(output_dir, exist_ok=True)
csv_output_path = os.path.join(output_dir, 'mitbih_beats_binary.csv')
df_beats.to_csv(csv_output_path, index=False)

print("\n✓ اكتملت المعالجة القياسية المجردة وحُفظ الملف في المسار بنجاح.")

جاري بدء معالجة السجلات وفقاً للمنهجية القياسية الموثقة...

✓ اكتملت المعالجة القياسية المجردة وحُفظ الملف في المسار بنجاح.


In [4]:
# =====================================================================
# الخطوة 5: التحقق الرقمي من البيانات المستخرجة ومقارنتها عمياء
# =====================================================================
import pandas as pd

# تحميل الملف الناتج حديثاً
csv_path = '/content/data/mitbih_beats_binary.csv'
df = pd.read_csv(csv_path)

# استخراج المؤشرات الستة المطلوبة في الخطة
total_beats = len(df)                       # 1. عدد النبضات الإجمالي
normal_count = (df['label'] == 0).sum()     # 2. عدد الطبيعي
abnormal_count = (df['label'] == 1).sum()   # 3. عدد غير الطبيعي
beat_length = df.shape[1] - 1               # 4. طول كل نبضة (باستثناء عمود اللبل)
num_columns = df.shape[1]                   # 5. عدد الأعمدة الإجمالي
data_shape = df.shape                       # 6. شكل البيانات (الأسطر والأعمدة)

print("="*40)
print("📊 تقرير التدقيق الرقمي للبيانات المستخرجة:")
print("="*40)
print(f"🔹 1. عدد النبضات الإجمالي: {total_beats:,}")
print(f"🔹 2. عدد النبضات الطبيعي (Class 0): {normal_count:,}")
print(f"🔹 3. عدد النبضات غير الطبيعي (Class 1): {abnormal_count:,}")
print(f"🔹 4. طول كل نبضة (Samples): {beat_length}")
print(f"🔹 5. عدد الأعمدة الإجمالي: {num_columns}")
print(f"🔹 6. شكل مصفوفة البيانات بالكامل: {data_shape}")
print("="*40)

📊 تقرير التدقيق الرقمي للبيانات المستخرجة:
🔹 1. عدد النبضات الإجمالي: 101,406
🔹 2. عدد النبضات الطبيعي (Class 0): 90,588
🔹 3. عدد النبضات غير الطبيعي (Class 1): 10,818
🔹 4. طول كل نبضة (Samples): 300
🔹 5. عدد الأعمدة الإجمالي: 301
🔹 6. شكل مصفوفة البيانات بالكامل: (101406, 301)


In [5]:
# =====================================================================
# سكربت التشريح الرقمي الشامل والتدقيق الإحصائي لقاعدة البيانات (Checklist)
# =====================================================================
import os
import wfdb
import numpy as np
import pandas as pd

raw_data_dir = '/content/mitdb_raw'
all_records = [
    100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
    111, 112, 113, 114, 115, 116, 117, 118, 119, 121,
    122, 123, 124, 200, 201, 202, 203, 205, 207, 208,
    209, 210, 212, 213, 214, 215, 217, 219, 220, 221,
    222, 223, 228, 230, 231, 232, 233, 234
]

# خريطة معايير AAMI EC57 الرسمية لرموز MIT-BIH
aami_mapping = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',  # الطبيعي وتأخير التوصيل
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',            # فوق البطينية Supraventricular
    'V': 'V', 'E': 'V',                                # البطينية Ventricular
    'F': 'F',                                          # الاندماج Fusion
    '/': 'Q', 'f': 'Q', 'Q': 'Q'                       # غير المعروف وطبياً لا تصنف ثنائياً
}

# قواميس لتخزين إحصائيات كل سجل طبي على حدة لملء الجداول بدقة
records_report = []
global_aami_counts = {'N': 0, 'S': 0, 'V': 0, 'F': 0, 'Q': 0}
global_removed_reasons = {'Boundary_Truncation': 0, 'Q_Exclusion': 0, 'Non_Medical_Symbols': 0}
total_raw_annotations = 0

print("🔍 جاري بدء الفحص والتشريح الرقمي الدقيق لـ 48 سجلاً طبياً...")

for rec in all_records:
    rec_str = str(rec)
    record_path = os.path.join(raw_data_dir, rec_str)

    try:
        # قراءة السجل والتحقق من اسم القناة الأولى المستخدمة
        record = wfdb.rdrecord(record_path)
        lead_name = record.sig_name[0] # استخراج اسم القناة الأولى المستخدمة فعلياً
        signal = record.p_signal[:, 0]
        annotation = wfdb.rdann(record_path, 'atr')
    except FileNotFoundError:
        print(f"⚠️ تحذير: ملفات السجل {rec_str} غير موجودة. جاري تخطي هذا السجل.")
        continue
    except Exception as e:
        print(f"⚠️ تحذير: حدث خطأ غير متوقع عند معالجة السجل {rec_str}: {e}. جاري تخطي هذا السجل.")
        continue

    symbols = annotation.symbol
    sample_locations = annotation.sample
    total_raw_annotations += len(symbols)

    # عدادات داخلية لكل سجل مريض
    rec_counts = {'Raw': len(symbols), 'N': 0, 'S': 0, 'V': 0, 'F': 0, 'Q': 0, 'Other': 0, 'Boundary_Drop': 0}
    rec_binary = {'Normal_Saved': 0, 'Abnormal_Saved': 0}

    for i in range(len(symbols)):
        sym = symbols[i]
        pos = sample_locations[i]

        if sym in aami_mapping:
            aami_cat = aami_mapping[sym]
            rec_counts[aami_cat] += 1
            global_aami_counts[aami_cat] += 1

            # تحقق من شرط الاقتطاع الطرفي (150 عينة قبل وبعد القمة)
            if pos - 150 >= 0 and pos + 150 < len(signal):
                if aami_cat == 'N':
                    rec_binary['Normal_Saved'] += 1
                elif aami_cat in ['S', 'V', 'F']:
                    rec_binary['Abnormal_Saved'] += 1
                else:
                    global_removed_reasons['Q_Exclusion'] += 1
            else:
                rec_counts['Boundary_Drop'] += 1
                global_removed_reasons['Boundary_Truncation'] += 1
        else:
            rec_counts['Other'] += 1
            global_removed_reasons['Non_Medical_Symbols'] += 1

    # حفظ تقرير تفصيلي لكل مريض (مفيد جداً لتوثيق الـ Pace ومسميات القنوات)
    records_report.append({
        'Record': rec,
        'Primary_Lead': lead_name,
        'Total_Annotations': rec_counts['Raw'],
        'AAMI_N': rec_counts['N'],
        'AAMI_S': rec_counts['S'],
        'AAMI_V': rec_counts['V'],
        'AAMI_F': rec_counts['F'],
        'AAMI_Q': rec_counts['Q'],
        'Non_Medical': rec_counts['Other'],
        'Boundary_Dropped': rec_counts['Boundary_Drop'],
        'Final_Normal_0': rec_binary['Normal_Saved'],
        'Final_Abnormal_1': rec_binary['Abnormal_Saved']
    })

# تحويل التقرير إلى DataFrame لعرضه بشكل مصفوفة منظمة
df_report = pd.DataFrame(records_report)

# طباعة التقرير العام والتحليل الإحصائي النهائي المباشر
print("\n" + "="*70)
print("📊 التقرير النهائي لتشريح وتدقيق قاعدة البيانات (Reviewers Checklist)")
print("="*70)
print(f"🔹 1. إجمالي التوصيفات الخام في السجلات (Total Raw Annotations): {total_raw_annotations:,}")
print(f"🔹 2. إجمالي النبضات التي نجحت وحُفظت فعلياً في ملف الـ CSV: {df_report['Final_Normal_0'].sum() + df_report['Final_Abnormal_1'].sum():,}")
print(f"     [- عدد النبضات الطبيعي المعتمد Class 0]: {df_report['Final_Normal_0'].sum():,}")
print(f"     [- عدد النبضات غير الطبيعي المعتمد Class 1]: {df_report['Final_Abnormal_1'].sum():,}")
print("-"*70)
print("📌 توزيع النبضات الخام الإجمالي حسب معايير AAMI (قبل أي اقتطاع أو حذف):")
print(f"  - فئة [N] (طبيعي وتأخير توصيل): {global_aami_counts['N']:,}")
print(f"  - فئة [S] (فوق بطيني مرضية):  {global_aami_counts['S']:,}")
print(f"  - فئة [V] (بطيني مرضية):      {global_aami_counts['V']:,}")
print(f"  - فئة [F] (اندماج مرضية):     {global_aami_counts['F']:,}")
print(f"  - فئة [Q] (غير معروفة طبياً):   {global_aami_counts['Q']:,}")
print("-"*70)
print("📌 التشريح الرياضي الدقيق لسبب استبعاد الـ 11 ألف نبضة الفارق:")
print(f"  - استبعاد بسبب النبضات الطرفية عند الحواف (Boundary Truncation): {global_removed_reasons['Boundary_Truncation']:,} نبضة")
print(f"  - استبعاد بسبب حظر فئة [Q] غير المعروفة طبياً (Q Exclusion): {global_removed_reasons['Q_Exclusion']:,} نبضة")
print(f"  - استبعاد بسبب رموز غير طبية ومساعدة (Non-Medical/Artifacts): {global_removed_reasons['Non_Medical_Symbols']:,} نبضة")
print("=" * 70)

# عرض ملخص لأول 5 سجلات طبية كعينة للتأكد من مسميات القنوات كـ Lead II و Paced Records
print("\n📋 عينة تفصيلية من السجلات (توضح القنوات الطبية المستخدمة فعلياً):")
print(df_report[['Record', 'Primary_Lead', 'Total_Annotations', 'Boundary_Dropped', 'Final_Normal_0', 'Final_Abnormal_1']].head(10).to_string(index=False))

🔍 جاري بدء الفحص والتشريح الرقمي الدقيق لـ 48 سجلاً طبياً...

📊 التقرير النهائي لتشريح وتدقيق قاعدة البيانات (Reviewers Checklist)
🔹 1. إجمالي التوصيفات الخام في السجلات (Total Raw Annotations): 112,647
🔹 2. إجمالي النبضات التي نجحت وحُفظت فعلياً في ملف الـ CSV: 101,406
     [- عدد النبضات الطبيعي المعتمد Class 0]: 90,588
     [- عدد النبضات غير الطبيعي المعتمد Class 1]: 10,818
----------------------------------------------------------------------
📌 توزيع النبضات الخام الإجمالي حسب معايير AAMI (قبل أي اقتطاع أو حذف):
  - فئة [N] (طبيعي وتأخير توصيل): 90,631
  - فئة [S] (فوق بطيني مرضية):  2,781
  - فئة [V] (بطيني مرضية):      7,236
  - فئة [F] (اندماج مرضية):     803
  - فئة [Q] (غير معروفة طبياً):   8,043
----------------------------------------------------------------------
📌 التشريح الرياضي الدقيق لسبب استبعاد الـ 11 ألف نبضة الفارق:
  - استبعاد بسبب النبضات الطرفية عند الحواف (Boundary Truncation): 50 نبضة
  - استبعاد بسبب حظر فئة [Q] غير المعروفة طبياً (Q Exclusion): 8,038 نبضة
  